In [15]:
print("hello");

hello


In [16]:
import os

for root, dirs, files in os.walk("C:\\Users\\91955"):
    if "Medical-Chatbot" in dirs:
        print("FOUND AT:", os.path.join(root, "Medical-Chatbot"))

FOUND AT: C:\Users\91955\Medical-Chatbot


In [17]:
%cd "C:\Users\91955\Medical-Chatbot"

C:\Users\91955\Medical-Chatbot


In [18]:
%pwd

'C:\\Users\\91955\\Medical-Chatbot'

In [19]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Extracted Document #

In [20]:
from langchain_community.document_loaders import PyPDFLoader
import os

def load_pdf_file(data):
    documents = []

    for file in os.listdir(data):
        if file.endswith(".pdf"):
            pdf_path = os.path.join(data, file)

            try:
                loader = PyPDFLoader(pdf_path)
                docs = loader.load()
                documents.extend(docs)

            except Exception as e:
                print(f"Error loading {file}: {e}")

    return documents


extracted_data = load_pdf_file("Data")
print(len(extracted_data))

759


In [21]:
extracted_data[0]

Document(metadata={'source': 'Data\\The_GALE_ENCYCLOPEDIA_of_MEDICINE_SECOND.pdf', 'page': 0}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

# Chunking #

In [22]:
def text_splitter(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
    split_docs = text_splitter.split_documents(extracted_data)
    return split_docs

In [52]:
split_docs=text_splitter(extracted_data)
print("length of split docs:",len(split_docs))

length of split docs: 3931


# Vector Embedding #

In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings

In [25]:
import sentence_transformers
print("sentence-transformers OK")
print(sentence_transformers.__version__)

sentence-transformers OK
5.5.1


In [26]:
def downloading_hugging_face_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2" #384
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return embeddings

In [27]:
embeddings = downloading_hugging_face_embeddings()

C:\Users\91955\AppData\Local\Temp\ipykernel_14408\3390227885.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4683.38it/s]


In [28]:
query = "What are the symptoms of diabetes?"
query_embedding = embeddings.embed_query(query)
print("length of query embedding:", len(query_embedding))

length of query embedding: 384


In [29]:
from dotenv import load_dotenv
load_dotenv()

True

In [30]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

In [31]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

# Initializing Pinecone #

In [35]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index = pc.Index("test")

index.delete(delete_all=True)

print("All vectors deleted")

All vectors deleted


In [36]:
print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {},
 'total_vector_count': 0}


In [37]:
from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_documents(
    documents=split_docs,
   index_name="test",
    embedding=embeddings
)
    


In [53]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name="test",
    embedding=embeddings
)

In [39]:
from pinecone import Pinecone
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index = pc.Index("test")

print(index.describe_index_stats())

{'dimension': 384,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 3931}},
 'total_vector_count': 3931}


In [40]:
results = docsearch.similarity_search(
    "What is diabetes?",
    k=2
)

for doc in results:
    print(doc.page_content)
    print("---------------")

glucose in the blood cannot be absorbed into the cells of
the body. Symptoms include frequent urination, lethargy,
excessive thirst, and hunger. The treatment includes
changes in diet, oral medications, and in some cases,
daily injections of insulin.
Description
Diabetes mellitus is a chronic disease that causes
serious health complications including renal (kidney)
failure, heart disease, stroke, and blindness. Approxi-
mately 14 million Americans (about 5% of the popula-
tion) have diabetes. Unfortunately, as many as one-half
are unaware that they have it.
Background
Every cell in the human body needs energy in order
to function. The body’s primary energy source is glu-
cose, a simple sugar resulting from the digestion of
foods containing carbohydrates (sugars and starches).
Glucose from the digested food circulates in the blood
as a ready energy source for any cells that need it.
Insulin is a hormone or chemical produced by cells in
the pancreas, an organ located behind the stomach.


In [41]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5
)

In [43]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [49]:
retriever = docsearch.as_retriever()
uestion_ans_chain=create_stuff_documents_chain(llm=llm, prompt=prompt)
rag_chain=create_retrieval_chain(retriever,question_ans_chain)

In [51]:
response=rag_chain.invoke({"input" : 'What is  diabetes?'})
print(response["answer"])

Diabetes is a chronic disease that causes the body to be unable to produce or respond properly to insulin, a hormone that converts glucose into energy. This leads to high levels of glucose in the blood, causing symptoms such as frequent urination, excessive thirst, and hunger.
